# S&P 500 Tail Risk Challenge — Instructor Reference Notebook

This notebook downloads Wikipedia's current constituent table and Yahoo Finance daily
**Adjusted Close** prices, calculates bias-corrected sample skewness and excess kurtosis,
applies the competition eligibility rules, and exports the reference leaderboard.

**Analysis dates:** 2024-01-02 through 2025-12-31, inclusive.
**Competition:** 2026-10-08 14:00 through 2026-10-12 23:59, Europe/Madrid; five attempts.

Workflow: configuration → frozen universe → exchange calendar → cached downloads →
consecutive-session returns → eligibility → equal-weight percentile score → audit exports.

**Instructor only:** outputs reveal the winning tickers. Do not distribute this executed
notebook or the reference leaderboard to students before the competition ends.

## Reproducibility and methodological limits

- Current constituents applied to an earlier period introduce survivorship and
  constituent-selection bias. This is a retrospective classroom screen, not a
  point-in-time backtest or a forecast. No future return observations enter the calculation.
- Provider adjustments and Wikipedia membership can change. Local input snapshots,
  SHA-256 hashes, parameters, package versions, and download timestamps are preserved.
  Re-running with the same cached inputs reproduces the calculations; downloading again
  may change them. Use a new output directory for a new competition snapshot.
- A high fourth moment may be dominated by very few observations. Negative skewness
  does not by itself establish higher potential losses. Equal weights are a pedagogical
  convention, not an estimated risk model. No Gaussian assumption is needed for ranking.
- Do not remove valid extreme returns or silently repair prices. Review unusually large
  moves and provider anomalies before freezing the scoring dataset.
- Yahoo Finance is an accessible educational data source, not a validated institutional
  feed. Check the provider's terms before redistributing its data.

## 1. Environment and configuration

Select the `.venv-tail-risk` Python interpreter in VS Code. For a different environment,
install the dependencies using the optional cell below, then restart the kernel.
No credentials or project `.env` files are used. Network access is required only to
create or explicitly retry input snapshots.

In [1]:
# Uncomment only if dependencies are missing from the selected kernel.
# %pip install numpy pandas scipy requests lxml yfinance exchange-calendars ipykernel

In [2]:
from datetime import datetime, timezone
from hashlib import sha256
from importlib.metadata import version
from io import StringIO
from pathlib import Path
import json
import math
import platform
import time

import exchange_calendars as xcals
import numpy as np
import pandas as pd
import requests
from scipy.stats import kurtosis, skew
import yfinance as yf
from IPython.display import display

START = pd.Timestamp('2024-01-02')
END = pd.Timestamp('2025-12-31')
MIN_COVERAGE = 0.90
KURTOSIS_WEIGHT = 0.50
SKEWNESS_WEIGHT = 0.50
BATCH_SIZE = 30
DOWNLOAD_ATTEMPTS = 3
# Explicitly enable this to retry failed symbols. Cached successful symbols stay frozen.
RETRY_FAILED_DOWNLOADS = False
WIKIPEDIA_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'docs' else cwd
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'competitions' / 'tail_risk_2024_2025'
CACHE_DIR = OUTPUT_DIR / 'price_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 20)

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def file_hash(path: Path) -> str:
    return sha256(path.read_bytes()).hexdigest()

assert START <= END
assert 0 < MIN_COVERAGE <= 1
assert math.isclose(KURTOSIS_WEIGHT + SKEWNESS_WEIGHT, 1.0)
print(f'Output directory: {OUTPUT_DIR}')

Output directory: C:\dev\sp500data\data\competitions\tail_risk_2024_2025


## 2. Download and freeze Wikipedia constituents

Preserve the source HTML and download metadata. Keep Wikipedia symbols for student
submissions; convert periods to hyphens only for Yahoo's price requests (for example,
`BRK.B` → `BRK-B`). Multiple share classes mean the table need not contain exactly
500 securities. The snapshot records the actual table returned, not historical membership.

In [3]:
html_path = OUTPUT_DIR / 'wikipedia_constituents.html'
universe_metadata_path = OUTPUT_DIR / 'universe_source.json'
if not html_path.exists():
    response = requests.get(
        WIKIPEDIA_URL,
        headers={'User-Agent': 'SP500TailRiskEducationalNotebook/1.0'},
        timeout=60,
    )
    response.raise_for_status()
    tables = pd.read_html(StringIO(response.text), attrs={'id': 'constituents'})
    if len(tables) != 1 or not {'Symbol', 'Security'}.issubset(tables[0].columns):
        raise RuntimeError('Wikipedia constituent table could not be identified.')
    html_path.write_text(response.text, encoding='utf-8')
    universe_metadata_path.write_text(json.dumps({
        'requested_url': WIKIPEDIA_URL,
        'resolved_url': response.url,
        'retrieved_at_utc': utc_now(),
        'html_sha256': file_hash(html_path),
    }, indent=2), encoding='utf-8')

universe_source = json.loads(universe_metadata_path.read_text(encoding='utf-8'))
if file_hash(html_path) != universe_source['html_sha256']:
    raise RuntimeError('The frozen Wikipedia HTML has changed.')
constituents = pd.read_html(
    StringIO(html_path.read_text(encoding='utf-8')), attrs={'id': 'constituents'}
)[0]
universe = constituents[['Symbol', 'Security', 'GICS Sector']].rename(columns={
    'Symbol': 'ticker', 'Security': 'company', 'GICS Sector': 'sector'
})
universe['ticker'] = universe['ticker'].astype(str).str.strip().str.upper()
universe['yahoo_ticker'] = universe['ticker'].str.replace('.', '-', regex=False)
universe = universe.sort_values('ticker').reset_index(drop=True)
if universe['ticker'].duplicated().any() or universe['yahoo_ticker'].duplicated().any():
    raise ValueError('Duplicate Wikipedia symbols or ambiguous Yahoo symbol mappings.')
if not 450 <= len(universe) <= 550:
    raise ValueError('Unexpected universe size; inspect the Wikipedia snapshot.')
universe.to_csv(OUTPUT_DIR / 'universe.csv', index=False)
print(f'Frozen universe: {len(universe)} securities')
print(f"Retrieved: {universe_source['retrieved_at_utc']}")
display(universe.head(10))

Frozen universe: 503 securities
Retrieved: 2026-10-07T07:57:48.631408+00:00


,ticker,company,sector,yahoo_ticker
0,A,Agilent Technologies,Health Care,A
1,AAPL,Apple Inc.,Information Technology,AAPL
2,ABBV,AbbVie,Health Care,ABBV
3,ABNB,Airbnb,Consumer Discretionary,ABNB
4,ABT,Abbott Laboratories,Health Care,ABT
5,ACGL,Arch Capital Group,Financials,ACGL
6,ACN,Accenture,Information Technology,ACN
7,ADBE,Adobe Inc.,Information Technology,ADBE
8,ADI,Analog Devices,Information Technology,ADI
9,ADM,Archer Daniels Midland,Consumer Staples,ADM


## 3. Define the trading-session denominator

Use the XNYS exchange calendar, shared by US listed equities for this exercise. It
excludes weekends, holidays and exceptional exchange closures. Short sessions count
as one trading day. An independent exchange calendar prevents provider outages from
reducing the coverage denominator.

Download the immediately preceding session solely to calculate the first in-period
return. The final date is inclusive; Yahoo's API end date is exclusive.

In [4]:
calendar = xcals.get_calendar('XNYS')
sessions = pd.DatetimeIndex(calendar.sessions_in_range(START, END)).tz_localize(None)
if sessions.empty:
    raise ValueError('No trading sessions in the requested period.')
previous_session = pd.Timestamp(calendar.previous_session(sessions[0])).tz_localize(None)
price_sessions = pd.DatetimeIndex([previous_session]).append(sessions)
expected_sessions = len(sessions)
minimum_valid_returns = math.ceil(MIN_COVERAGE * expected_sessions)
download_start = previous_session.strftime('%Y-%m-%d')
download_end = (END + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
pd.Series(sessions, name='session').to_csv(OUTPUT_DIR / 'trading_sessions.csv', index=False)
print(f'Expected sessions: {expected_sessions}')
print(f'Minimum valid daily returns: {minimum_valid_returns}')
print(f'Pre-period price date: {previous_session.date()}')

Expected sessions: 502


Minimum valid daily returns: 452
Pre-period price date: 2023-12-29


## 4. Download daily Adjusted Close prices with caching and retry logs

`auto_adjust=False` explicitly requests the provider's `Adj Close` field; we never
substitute unadjusted Close. Downloads use small batches and bounded retries. Each
successful symbol is cached separately with retrieval metadata and a checksum.
Partial histories are retained and evaluated by the 90% rule, not artificially filled.

Symbols with no usable history are recorded as **download failures**, separately from
insufficient coverage or non-negative skewness. This notebook can produce a diagnostic
ranking despite failed symbols, but it labels that ranking provisional. Resolve failures
before importing reference scores into Moodle. Set `RETRY_FAILED_DOWNLOADS=True` to
retry them on a later run.

In [5]:
def extract_adjusted_close(raw: pd.DataFrame, symbols: list[str]) -> pd.DataFrame:
    if raw is None or raw.empty:
        return pd.DataFrame(columns=symbols, dtype=float)
    if isinstance(raw.columns, pd.MultiIndex):
        if 'Adj Close' not in raw.columns.get_level_values(0):
            return pd.DataFrame(columns=symbols, dtype=float)
        panel = raw.xs('Adj Close', axis=1, level=0).copy()
    elif len(symbols) == 1 and 'Adj Close' in raw.columns:
        panel = raw[['Adj Close']].rename(columns={'Adj Close': symbols[0]})
    else:
        return pd.DataFrame(columns=symbols, dtype=float)
    panel.index = pd.DatetimeIndex(panel.index).tz_localize(None).normalize()
    if panel.index.has_duplicates:
        raise ValueError('Duplicate daily timestamps returned by the provider.')
    return panel.sort_index().reindex(columns=symbols).apply(pd.to_numeric, errors='coerce')

def download_batch(symbols: list[str]) -> pd.DataFrame:
    raw = yf.download(
        symbols, start=download_start, end=download_end, interval='1d',
        auto_adjust=False, back_adjust=False, repair=False, keepna=True,
        actions=False, group_by='column', threads=4, progress=False,
        ignore_tz=True, timeout=30,
    )
    return extract_adjusted_close(raw, symbols)

metadata_path = OUTPUT_DIR / 'price_download_metadata.json'
download_metadata = json.loads(metadata_path.read_text(encoding='utf-8')) if metadata_path.exists() else {}
request_signature = {'start': download_start, 'end_exclusive': download_end,
                     'interval': '1d', 'auto_adjust': False, 'repair': False}
prices_by_ticker = {}
pending = []
for row in universe.itertuples(index=False):
    cache_path = CACHE_DIR / f'{row.yahoo_ticker}.csv'
    record = download_metadata.get(row.ticker)
    if record is not None and record['request'] != request_signature:
        raise ValueError('Cached download parameters differ; choose a new output directory.')
    if cache_path.exists():
        if record is None or record.get('sha256') != file_hash(cache_path):
            raise ValueError(f'Unverified price cache for {row.ticker}.')
        cached = pd.read_csv(cache_path, index_col='date', parse_dates=True)
        prices_by_ticker[row.ticker] = cached['adj_close']
    elif record and record['status'] in {'failed', 'ineligible_listing_history'} and not RETRY_FAILED_DOWNLOADS:
        prices_by_ticker[row.ticker] = pd.Series(dtype=float)
    else:
        pending.append(row)

print(f'Cached symbols: {len(prices_by_ticker)}; symbols to download: {len(pending)}', flush=True)
for offset in range(0, len(pending), BATCH_SIZE):
    remaining = pending[offset:offset + BATCH_SIZE]
    errors = {}
    for attempt in range(1, DOWNLOAD_ATTEMPTS + 1):
        if not remaining:
            break
        symbols = [row.yahoo_ticker for row in remaining]
        try:
            panel = download_batch(symbols)
        except Exception as exc:
            panel = pd.DataFrame(columns=symbols, dtype=float)
            errors.update({row.ticker: f'{type(exc).__name__}: {exc}' for row in remaining})
        failed = []
        for row in remaining:
            series = panel[row.yahoo_ticker] if row.yahoo_ticker in panel else pd.Series(dtype=float)
            usable = series.reindex(price_sessions)
            if not (np.isfinite(usable) & (usable > 0)).any():
                errors.setdefault(row.ticker, 'No finite positive Adjusted Close on requested sessions.')
                failed.append(row)
                continue
            series = series.rename('adj_close')
            series.index.name = 'date'
            cache_path = CACHE_DIR / f'{row.yahoo_ticker}.csv'
            series.to_csv(cache_path, float_format='%.17g')
            prices_by_ticker[row.ticker] = pd.read_csv(
                cache_path, index_col='date', parse_dates=True
            )['adj_close']
            download_metadata[row.ticker] = {
                'status': 'ok', 'yahoo_ticker': row.yahoo_ticker,
                'retrieved_at_utc': utc_now(), 'request': request_signature,
                'sha256': file_hash(cache_path), 'nonmissing_rows': int(series.notna().sum()),
            }
        remaining = failed
        if remaining and attempt < DOWNLOAD_ATTEMPTS:
            time.sleep(2 * attempt)
    for row in remaining:
        prices_by_ticker[row.ticker] = pd.Series(dtype=float)
        download_metadata[row.ticker] = {
            'status': 'failed', 'yahoo_ticker': row.yahoo_ticker,
            'retrieved_at_utc': utc_now(), 'request': request_signature,
            'error': errors.get(row.ticker, 'Unresolved download failure.'),
        }
    metadata_path.write_text(json.dumps(download_metadata, indent=2), encoding='utf-8')
    print(f'Processed {min(offset + BATCH_SIZE, len(pending))}/{len(pending)} new symbols', flush=True)
    time.sleep(0.5)

# Resolve failed requests only when a documented listing date proves ineligibility.
# Do not splice predecessor securities across mergers or spin-offs.
listing_evidence = {
    'FDXF': {
        'first_regular_trading_date': '2026-06-01',
        'source': 'https://investor.fedex.com/news-and-events/investor-news/investor-news-details/2026/FedEx-Completes-Spin-Off-of-FedEx-Freight/default.aspx',
    },
    'HONA': {
        'first_regular_trading_date': '2026-06-29',
        'source': 'https://www.honeywell.com/us/en/news/press-releases/2026/06/honeywell-technologies-launches-independent-pure-play-automation-company-following-honeywell-aerospace-spin-off',
    },
    'PSKY': {
        'first_regular_trading_date': '2025-08-07',
        'source': 'https://www.paramount.com/press/skydance-media-and-paramount-global-complete-merger-creating-next-generation-media-company',
    },
}
for ticker, evidence in listing_evidence.items():
    record = download_metadata.get(ticker)
    if record is None or record['status'] not in {'failed', 'ineligible_listing_history'}:
        continue
    listed = pd.Timestamp(evidence['first_regular_trading_date'])
    # A conservative upper bound: even counting the first listing day as a valid return.
    possible_sessions = int((sessions >= listed).sum())
    if possible_sessions < minimum_valid_returns:
        record.update({
            'status': 'ineligible_listing_history', 'listing_evidence': evidence,
            'maximum_possible_coverage': possible_sessions / expected_sessions,
        })
metadata_path.write_text(json.dumps(download_metadata, indent=2), encoding='utf-8')
pd.DataFrame([
    {'ticker': ticker, **record['listing_evidence'],
     'maximum_possible_coverage': record['maximum_possible_coverage']}
    for ticker, record in download_metadata.items() if record['status'] == 'ineligible_listing_history'
]).to_csv(OUTPUT_DIR / 'confirmed_listing_exclusions.csv', index=False)

adjusted_close = pd.DataFrame(prices_by_ticker).reindex(
    index=price_sessions, columns=universe['ticker'].tolist()
)
download_failures = pd.DataFrame([
    {'ticker': ticker, **record}
    for ticker, record in download_metadata.items() if record['status'] == 'failed'
], columns=['ticker', 'status', 'yahoo_ticker', 'retrieved_at_utc', 'request', 'error'])
adjusted_close.loc[START:END].to_csv(OUTPUT_DIR / 'adjusted_close_daily.csv', float_format='%.17g')
adjusted_close.to_csv(OUTPUT_DIR / 'adjusted_close_with_seed.csv', float_format='%.17g')
download_failures.to_csv(OUTPUT_DIR / 'download_failures.csv', index=False)
READY_FOR_FREEZE = download_failures.empty
print(f'Unresolved download failures: {len(download_failures)}')
print(f'Ready for a statistical audit before freezing: {READY_FOR_FREEZE}')
if not download_failures.empty:
    display(download_failures[['ticker', 'error']])

Cached symbols: 503; symbols to download: 0


Unresolved download failures: 0
Ready for a statistical audit before freezing: True


## 5. Consecutive-session returns and eligibility

Reindex prices to the complete session grid **before** computing
$r_t=\log(P_t)-\log(P_{t-1})$. Missing, non-finite and non-positive prices invalidate
the affected returns. A missing price invalidates both its own return and the next
session's return. No forward fill, interpolation, or multi-session gap bridging occurs.

FDXF and HONA began regular trading in 2026; PSKY is a new post-merger security
first traded in August 2025. Their issuer announcements, linked in the download metadata
and `confirmed_listing_exclusions.csv`, establish a maximum possible coverage below
90%. No predecessor histories are spliced into these securities. For these failed price
requests, zero observed returns is a download result, not a claim that no trading occurred;
the documented coverage upper bound is what independently establishes exclusion.

Coverage is the number of valid **returns** divided by all scheduled sessions in the
analysis period, not the proportion of nonmissing prices. The seed date is excluded
from the coverage denominator. The threshold includes equality at 90%.

Use `skew(..., bias=False)` and `kurtosis(..., fisher=True, bias=False)`; the latter
reports excess kurtosis (zero for the normal population). Zero-variance samples and
samples with fewer than four valid observations have undefined competition metrics.
Eligibility additionally requires strictly negative skewness.

In [6]:
def calculate_returns(prices: pd.DataFrame, analysis_sessions: pd.DatetimeIndex) -> pd.DataFrame:
    clean = prices.where(np.isfinite(prices) & (prices > 0))
    return np.log(clean).diff().reindex(analysis_sessions).replace([np.inf, -np.inf], np.nan)

def calculate_metrics(returns: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for ticker in returns.columns:
        values = returns[ticker].dropna().to_numpy(dtype=float)
        n_valid = len(values)
        defined = n_valid >= 4 and np.ptp(values) > 0
        rows.append({
            'ticker': ticker, 'n_valid_returns': n_valid,
            'expected_sessions': len(returns), 'coverage': n_valid / len(returns),
            'skewness': float(skew(values, bias=False)) if defined else np.nan,
            'excess_kurtosis': float(kurtosis(values, fisher=True, bias=False)) if defined else np.nan,
            'daily_volatility': float(np.std(values, ddof=1)) if n_valid >= 2 else np.nan,
            'minimum_daily_return': float(values.min()) if n_valid else np.nan,
            'maximum_daily_return': float(values.max()) if n_valid else np.nan,
        })
    return pd.DataFrame(rows)

returns = calculate_returns(adjusted_close, sessions)
metrics = universe.merge(calculate_metrics(returns), on='ticker', validate='one_to_one')
metrics['download_status'] = metrics['ticker'].map(lambda ticker: download_metadata[ticker]['status'])
metrics['maximum_possible_coverage_from_listing'] = metrics['ticker'].map(
    lambda ticker: download_metadata[ticker].get('maximum_possible_coverage', np.nan)
)
metrics['coverage_ok'] = metrics['n_valid_returns'] >= minimum_valid_returns
metrics['moments_defined'] = np.isfinite(metrics['skewness']) & np.isfinite(metrics['excess_kurtosis'])
metrics['negative_skewness'] = metrics['skewness'] < 0
metrics['eligible'] = (
    metrics['download_status'].eq('ok') & metrics['coverage_ok']
    & metrics['moments_defined'] & metrics['negative_skewness']
)

def exclusion_reasons(row: pd.Series) -> str:
    reasons = []
    if row['download_status'] == 'ineligible_listing_history':
        reasons.append('documented_listing_history_cannot_reach_90_percent')
    elif row['download_status'] != 'ok':
        reasons.append('download_failed_unresolved')
    if not row['coverage_ok']:
        reasons.append('valid_return_coverage_below_90_percent')
    if not row['moments_defined']:
        reasons.append('undefined_moments')
    elif not row['negative_skewness']:
        reasons.append('skewness_not_strictly_negative')
    return '; '.join(reasons)

metrics['exclusion_reasons'] = metrics.apply(exclusion_reasons, axis=1)
returns.to_csv(OUTPUT_DIR / 'daily_log_returns.csv', float_format='%.17g')
print(f'Universe: {len(metrics)} | Eligible: {metrics.eligible.sum()} | Excluded: {(~metrics.eligible).sum()}')
display(metrics.groupby(['download_status', 'coverage_ok', 'negative_skewness']).size().rename('stocks'))

Universe: 503 | Eligible: 332 | Excluded: 171


download_status             coverage_ok  negative_skewness
ineligible_listing_history  False        False                  3
ok                          False        False                  2
                                         True                   3
                            True         False                163
                                         True                 332
Name: stocks, dtype: int64

## 6. Equal-weight combined score and deterministic leaderboard

Among eligible stocks, use pandas average ranks divided by the eligible sample size:
$P(x_i)=\operatorname{rank}_{average}(x_i)/N$. Higher kurtosis and more negative skewness
yield higher ranks. Tied observations receive equal average ranks.

$C_i=0.5P(K_i)+0.5P(-S_i)$ and $G_i=10C_i/\max_j C_j$.

Grade 10 is awarded to every joint winner. Rank ties use competition ranks (1, 1, 3);
alphabetical order is for display only. These are full-precision reference grades,
**not yet rounded to Moodle's accepted answer fractions**. No Moodle XML is generated
in this notebook. Resolve any download failures before using the leaderboard for scoring.

In [7]:
def rank_eligible(table: pd.DataFrame) -> pd.DataFrame:
    ranked = table.loc[table['eligible']].copy()
    if ranked.empty:
        raise ValueError('No eligible stocks; inspect the download and coverage audit.')
    ranked['kurtosis_percentile'] = ranked['excess_kurtosis'].rank(method='average', pct=True)
    ranked['negative_skewness_percentile'] = (-ranked['skewness']).rank(method='average', pct=True)
    ranked['combined_score'] = (
        KURTOSIS_WEIGHT * ranked['kurtosis_percentile']
        + SKEWNESS_WEIGHT * ranked['negative_skewness_percentile']
    )
    best_score = ranked['combined_score'].max()
    ranked['distance_to_best_score'] = best_score - ranked['combined_score']
    ranked['grade_out_of_10'] = 10 * ranked['combined_score'] / best_score
    ranked['competition_rank'] = ranked['combined_score'].rank(ascending=False, method='min').astype(int)
    return ranked.sort_values(['combined_score', 'ticker'], ascending=[False, True]).reset_index(drop=True)

leaderboard = rank_eligible(metrics)
leaderboard['reference_status'] = 'audit_required' if READY_FOR_FREEZE else 'provisional_download_failures'
display(leaderboard[['competition_rank', 'ticker', 'company', 'coverage', 'excess_kurtosis',
                     'skewness', 'combined_score', 'distance_to_best_score', 'grade_out_of_10']].head(20))

,competition_rank,ticker,company,coverage,excess_kurtosis,skewness,combined_score,distance_to_best_score,grade_out_of_10
0,1,GL,Globe Life,1.000000,280.999709,-14.196996,1.000000,0.000000,10.000000
1,2,FISV,Fiserv,0.996016,196.182838,-11.872969,0.996988,0.003012,9.969880
2,3,EW,Edwards Lifesciences,1.000000,139.148558,-8.178498,0.993976,0.006024,9.939759
3,4,CNC,Centene Corporation,1.000000,124.828369,-7.933332,0.990964,0.009036,9.909639
4,5,DXCM,Dexcom,1.000000,101.555263,-6.839502,0.987952,0.012048,9.879518
5,6,ALGN,Align Technology,1.000000,91.427183,-6.125455,0.983434,0.016566,9.834337
6,7,WST,West Pharmaceutical Services,1.000000,97.250321,-5.993032,0.981928,0.018072,9.819277
7,8,IT,Gartner,1.000000,85.032703,-6.037778,0.980422,0.019578,9.804217
8,9,SNPS,Synopsys,1.000000,74.904511,-5.334732,0.974398,0.025602,9.743976
9,10,DG,Dollar General,1.000000,76.889853,-4.934843,0.971386,0.028614,9.713855


## 7. Verification and extreme-return audit

These assertions check the methodological boundaries: missing prices must not produce
gap-spanning returns, exactly 90% coverage passes, zero skewness fails, ties retain the
same score, and inputs contain no return dates outside the analysis window.

The largest absolute returns are exported for a manual corporate-action and data-quality
review. Keep valid extremes; correct genuine data errors only through an explicitly
documented replacement reference dataset and recalculate all scores consistently.

In [8]:
toy_dates = pd.date_range('2024-01-01', periods=5)
toy_prices = pd.DataFrame({'X': [100, 110, np.nan, 120, 126]}, index=toy_dates)
toy_returns = calculate_returns(toy_prices, toy_dates[1:])['X']
assert np.isclose(toy_returns.iloc[0], np.log(1.1))
assert toy_returns.iloc[1:3].isna().all()
assert np.isclose(toy_returns.iloc[3], np.log(1.05))
assert 90 >= math.ceil(0.90 * 100) and not 89 >= math.ceil(0.90 * 100)
assert not (0.0 < 0)
constant_metrics = calculate_metrics(pd.DataFrame({'X': np.zeros(10)}))
assert constant_metrics[['skewness', 'excess_kurtosis']].isna().all().all()
tied = rank_eligible(pd.DataFrame({
    'ticker': ['A', 'B'], 'eligible': [True, True],
    'excess_kurtosis': [5.0, 5.0], 'skewness': [-1.0, -1.0],
}))
assert tied['combined_score'].nunique() == 1 and tied['grade_out_of_10'].eq(10).all()
assert returns.index.equals(sessions)
assert metrics['n_valid_returns'].between(0, expected_sessions).all()
assert leaderboard['coverage'].ge(MIN_COVERAGE).all()
assert leaderboard['skewness'].lt(0).all()
assert np.isfinite(leaderboard[['skewness', 'excess_kurtosis', 'combined_score']]).all().all()
assert np.isclose(leaderboard['grade_out_of_10'].max(), 10)
assert len(metrics) == len(universe)
# Independently verify SciPy's bias-corrected moment estimates for the leader.
leader_ticker = leaderboard.loc[0, 'ticker']
observed = returns[leader_ticker].dropna().to_numpy()
n = len(observed)
centered = observed - observed.mean()
m2, m3, m4 = [np.mean(centered ** order) for order in (2, 3, 4)]
manual_skewness = np.sqrt(n * (n - 1)) / (n - 2) * m3 / m2 ** 1.5
raw_excess = m4 / m2 ** 2 - 3
manual_kurtosis = (n - 1) / ((n - 2) * (n - 3)) * ((n + 1) * raw_excess + 6)
assert np.isclose(manual_skewness, leaderboard.loc[0, 'skewness'], rtol=1e-10)
assert np.isclose(manual_kurtosis, leaderboard.loc[0, 'excess_kurtosis'], rtol=1e-10)
extreme_returns = returns.rename_axis('date').stack().rename('log_return').reset_index()
extreme_returns.columns = ['date', 'ticker', 'log_return']
extreme_returns['absolute_log_return'] = extreme_returns['log_return'].abs()
extreme_returns = extreme_returns.sort_values(
    ['absolute_log_return', 'ticker', 'date'], ascending=[False, True, True]
)
extreme_returns.head(100).to_csv(OUTPUT_DIR / 'extreme_returns_audit.csv', index=False)
print('Methodological checks passed. Review the extreme-return audit before freezing.')
display(extreme_returns.head(15))

Methodological checks passed. Review the extreme-return audit before freezing.


,date,ticker,log_return,absolute_log_return
34392,2024-04-11,GL,-0.758010,0.758010
228274,2025-10-29,FISV,-0.580600,0.580600
205782,2025-08-26,ECHO,0.532084,0.532084
70690,2024-07-26,DXCM,-0.521858,0.521858
186769,2025-07-02,CNC,-0.517023,0.517023
139751,2025-02-13,WST,-0.481563,0.481563
109939,2024-11-15,BE,0.464908,0.464908
196670,2025-07-31,ALGN,-0.456120,0.456120
211027,2025-09-10,SNPS,-0.443748,0.443748
181331,2025-06-16,ECHO,0.399509,0.399509


## 8. Export reference tables, feedback and reproducibility manifest

Exports include the entire universe with exclusion reasons, eligible rankings, daily
prices and returns, download failures, and a Moodle preparation table. Failed downloads
receive an unresolved status rather than a fabricated zero grade. Other ineligible
stocks receive zero, as specified in the competition rules.

Preserve this directory after review. Students should receive the reference price data
if exact reproducibility across data providers is required, but not the answer tables.
The manifest records sources, dates, definitions, dependency versions and input hashes.

In [9]:
score_columns = ['ticker', 'competition_rank', 'kurtosis_percentile', 'negative_skewness_percentile',
                 'combined_score', 'distance_to_best_score', 'grade_out_of_10']
reference = metrics.merge(leaderboard[score_columns], on='ticker', how='left', validate='one_to_one')
reference.loc[~reference['eligible'] & reference['download_status'].ne('failed'), 'grade_out_of_10'] = 0.0
reference['reference_status'] = 'audit_required' if READY_FOR_FREEZE else 'provisional_download_failures'

def make_feedback(row: pd.Series) -> str:
    if row['download_status'] == 'ineligible_listing_history':
        return (
            'Ineligible asset: its documented listing history cannot reach 90% coverage. '
            f"Maximum possible coverage: {row['maximum_possible_coverage_from_listing']:.2%}. Score: 0/10."
        )
    if row['download_status'] != 'ok':
        return 'Reference data unresolved. Do not use this answer for live grading yet.'
    if not row['eligible']:
        return f"Ineligible asset. Reason: {row['exclusion_reasons']}. Score: 0/10."
    return (
        f"Excess kurtosis: {row['excess_kurtosis']:.6f}. "
        f"Skewness: {row['skewness']:.6f}. "
        f"Valid return coverage: {row['coverage']:.2%}. "
        f"Combined score: {row['combined_score']:.6f}. "
        f"Distance to best score: {row['distance_to_best_score']:.6f}. "
        f"Reference grade: {row['grade_out_of_10']:.4f}/10."
    )

reference['feedback'] = reference.apply(make_feedback, axis=1)
reference.to_csv(OUTPUT_DIR / 'all_assets_metrics.csv', index=False, float_format='%.17g')
leaderboard.to_csv(OUTPUT_DIR / 'eligible_leaderboard.csv', index=False, float_format='%.17g')
reference.loc[~reference['eligible']].to_csv(OUTPUT_DIR / 'excluded_assets.csv', index=False)
reference[['ticker', 'eligible', 'grade_out_of_10', 'feedback', 'reference_status']].to_csv(
    OUTPUT_DIR / 'moodle_reference.csv', index=False
)
packages = ['numpy', 'pandas', 'scipy', 'requests', 'lxml', 'yfinance', 'exchange-calendars']
package_versions = {name: version(name) for name in packages}
manifest = {
    'generated_at_utc': utc_now(), 'python': platform.python_version(),
    'analysis_start': str(START.date()), 'analysis_end_inclusive': str(END.date()),
    'previous_price_session': str(previous_session.date()),
    'competition_open': '2026-10-08T14:00:00+02:00',
    'competition_close': '2026-10-12T23:59:00+02:00', 'competition_timezone': 'Europe/Madrid',
    'maximum_attempts': 5, 'calendar': 'XNYS', 'expected_sessions': expected_sessions,
    'minimum_valid_returns': minimum_valid_returns, 'minimum_coverage': MIN_COVERAGE,
    'prices': 'Yahoo Finance Adj Close, auto_adjust=False, repair=False',
    'returns': 'natural-log differences of consecutive exchange-session adjusted prices',
    'skewness': 'scipy.stats.skew(bias=False)',
    'kurtosis': 'scipy.stats.kurtosis(fisher=True, bias=False)',
    'eligibility': 'coverage >= 0.90; skewness < 0; finite moments; successful download',
    'percentile_definition': 'pandas rank(method=average, pct=True) among eligible assets',
    'weights': {'kurtosis': KURTOSIS_WEIGHT, 'negative_skewness': SKEWNESS_WEIGHT},
    'grade_definition': '10 * combined_score / maximum_eligible_combined_score',
    'universe_size': len(universe), 'eligible_assets': int(metrics['eligible'].sum()),
    'download_failures': len(download_failures),
    'listing_evidence': listing_evidence,
    'status': 'audit_required' if READY_FOR_FREEZE else 'provisional_download_failures',
    'universe_source': universe_source, 'packages': package_versions,
    'price_cache_hashes': {ticker: record.get('sha256') for ticker, record in download_metadata.items()},
    'file_hashes': {p.name: file_hash(p) for p in sorted(OUTPUT_DIR.glob('*.csv'))},
}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'requirements-lock.txt').write_text(
    '\n'.join(f'{name}=={v}' for name, v in package_versions.items()) + '\n', encoding='utf-8'
)
display(reference.loc[reference['eligible'], ['ticker', 'grade_out_of_10', 'feedback']].sort_values(
    ['grade_out_of_10', 'ticker'], ascending=[False, True]
).head(5))
print(f'Exported all reference artifacts to {OUTPUT_DIR}')
print(f"Reference status: {manifest['status']}")

,ticker,grade_out_of_10,feedback
206,GL,10.000000,Excess kurtosis: 280.999709. Skewness: -14.196...
188,FISV,9.969880,Excess kurtosis: 196.182838. Skewness: -11.872...
170,EW,9.939759,Excess kurtosis: 139.148558. Skewness: -8.1784...
100,CNC,9.909639,Excess kurtosis: 124.828369. Skewness: -7.9333...
149,DXCM,9.879518,Excess kurtosis: 101.555263. Skewness: -6.8395...


Exported all reference artifacts to C:\dev\sp500data\data\competitions\tail_risk_2024_2025
Reference status: audit_required


## Sources and interpretation

- [Wikipedia constituent table](https://en.wikipedia.org/wiki/List_of_S%26P_500_companies)
- [yfinance download parameters](https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html)
- [Exchange calendars and session definitions](https://github.com/gerrymanoim/exchange_calendars)
- [SciPy skewness](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.skew.html)
- [SciPy kurtosis](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.kurtosis.html)

This notebook establishes in-sample contest answers. It does not validate a risk forecast.
Any later claim about predictive risk performance requires an independent out-of-sample
period, a point-in-time universe and appropriate uncertainty analysis. Do not interpret
an empirical percentile ranking as a probability of loss or as a normality test.